# Chapter 12: DPO and GRPO — Alignment Without PPO

*Build a Multimodal Model from Scratch · Chapter 12*

> **This chapter in context:** Ch11 trained a reward model that scores (image, caption) pairs. The natural next step is to use that score to improve the VLM — but PPO (the classic RL algorithm for this) is notoriously unstable and requires 4 models running simultaneously. This chapter shows two simpler alternatives: DPO (no RL loop, no reward model) and GRPO (no reward model, just a verifier). Together these cover the full modern alignment toolkit.

In [ ]:
%matplotlib inline
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt, copy
from torch.utils.data import Dataset, DataLoader
import math
torch.manual_seed(42); np.random.seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PAL = dict(bg='#F8F9FA', dark='#2C3E50', blue='#4A90D9', orange='#E67E22',
           green='#27AE60', red='#E74C3C', purple='#8E44AD', grey='#95A5A6')
VOCAB = ["<pad>","<bos>","<eos>","a","red","blue","green","yellow",
         "circle","square","triangle","star","correct","wrong","good","bad"]
w2i = {w:i for i,w in enumerate(VOCAB)}; i2w = {i:w for i,w in enumerate(VOCAB)}
CAPTIONS = [["<bos>","a","red","circle","<eos>"],["<bos>","a","blue","square","<eos>"],
            ["<bos>","a","green","triangle","<eos>"],["<bos>","a","yellow","star","<eos>"]]
print(f"Device: {DEVICE}")

## Problem 1 — Why Not Just Use PPO?

Before DPO and GRPO, the standard alignment recipe was **RLHF with PPO**. Here is why that is painful:

**The PPO pipeline for RLHF requires 4 models in memory simultaneously:**
1. **Reference policy** (frozen SFT copy) — computes KL baseline
2. **Policy** (trainable) — the model being aligned
3. **Value model** — estimates expected future reward (a separate network)
4. **Reward model** — scores each completion

**Why PPO is unstable:**
- The RL training loop has its own inner optimisation loop (the "trust-region" update)
- KL budget, clipping coefficient, value loss weight — all need careful tuning
- Reward hacking is common: the policy finds ways to maximise reward that are not aligned with human intent
- Gradient variance from the RL objective is much higher than supervised loss

**Complexity:** The PPO training loop is 3× more code than SFT, requires synchronised rollout generation and value estimation, and typically needs 8× the VRAM of SFT.

The diagram below compares the three pipelines covered in this chapter.

In [ ]:
def draw_comparison_diagram():
    fig, ax = plt.subplots(figsize=(13, 6))
    fig.patch.set_facecolor(PAL['bg'])
    ax.set_facecolor(PAL['bg'])
    ax.set_xlim(0, 13); ax.set_ylim(0, 6)
    ax.axis('off')

    def box(x, y, w, h, color, text, fontsize=9, text_color='white'):
        rect = plt.Rectangle((x, y), w, h, facecolor=color, edgecolor=PAL['dark'],
                              linewidth=1.2, zorder=2)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h/2, text, ha='center', va='center',
                fontsize=fontsize, color=text_color, fontweight='bold', zorder=3,
                wrap=True)

    def arrow(x1, y1, x2, y2, color=PAL['dark']):
        ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle='->', color=color, lw=1.5), zorder=3)

    # ---- Row 1: PPO ----
    ax.text(0.15, 4.65, 'PPO', fontsize=11, fontweight='bold', color=PAL['red'], va='center')
    box(0.9, 4.3, 1.4, 0.7, PAL['grey'], 'SFT Model', 9)
    arrow(2.3, 4.65, 2.7, 4.65)
    box(2.7, 4.3, 1.5, 0.7, PAL['purple'], 'Reward\nModel', 8)
    arrow(4.2, 4.65, 4.6, 4.65)
    box(4.6, 4.3, 4.2, 0.7, PAL['red'], 'PPO Loop: policy + value + ref + RM', 8)
    arrow(8.8, 4.65, 9.2, 4.65)
    box(9.2, 4.3, 1.8, 0.7, PAL['dark'], 'Aligned\nModel', 9)
    ax.text(11.1, 4.65, '4 models\nUNSTABLE', fontsize=8, color=PAL['red'],
            va='center', fontweight='bold')

    # ---- Row 2: DPO ----
    ax.text(0.15, 3.05, 'DPO', fontsize=11, fontweight='bold', color=PAL['blue'], va='center')
    box(0.9, 2.7, 1.4, 0.7, PAL['grey'], 'SFT Model', 9)
    arrow(2.3, 3.05, 2.7, 3.05)
    box(2.7, 2.7, 3.5, 0.7, PAL['blue'], 'DPO: policy + ref\n(preference pairs)', 8)
    arrow(6.2, 3.05, 9.2, 3.05)
    box(9.2, 2.7, 1.8, 0.7, PAL['dark'], 'Aligned\nModel', 9)
    ax.text(11.1, 3.05, '2 models\nStable', fontsize=8, color=PAL['blue'],
            va='center', fontweight='bold')

    # ---- Row 3: GRPO ----
    ax.text(0.15, 1.45, 'GRPO', fontsize=11, fontweight='bold', color=PAL['orange'], va='center')
    box(0.9, 1.1, 1.4, 0.7, PAL['grey'], 'SFT Model', 9)
    arrow(2.3, 1.45, 2.7, 1.45)
    box(2.7, 1.1, 3.5, 0.7, PAL['orange'], 'GRPO: policy + verifier\n(multi-sample)', 8)
    arrow(6.2, 1.45, 9.2, 1.45)
    box(9.2, 1.1, 1.8, 0.7, PAL['dark'], 'Aligned\nModel', 9)
    ax.text(11.1, 1.45, '1 model\n+verifier', fontsize=8, color=PAL['orange'],
            va='center', fontweight='bold')

    # Complexity indicator legend
    ax.text(6.5, 5.6, 'Complexity / Memory', fontsize=9, color=PAL['grey'],
            ha='center', style='italic')
    for i, (label, color) in enumerate([('High (PPO)', PAL['red']),
                                         ('Medium (DPO)', PAL['blue']),
                                         ('Low (GRPO)', PAL['orange'])]):
        ax.plot([5.2 + i*1.4], [5.25], 's', color=color, ms=10)
        ax.text(5.35 + i*1.4, 5.25, label, fontsize=8, color=color, va='center')

    ax.set_title('Three Alignment Pipelines: PPO vs DPO vs GRPO',
                 fontsize=13, fontweight='bold', color=PAL['dark'], pad=12)
    plt.tight_layout()
    plt.show()

draw_comparison_diagram()

In [ ]:
def make_image(cls, size=32, noise=0.05):
    img = np.zeros((size, size, 3), dtype=np.float32)
    colors = [(1,0,0),(0,0,1),(0,0.7,0),(1,0.8,0)]
    c = colors[cls]; h,w,r = size//2,size//2,size//4
    if cls == 0:
        for i in range(size):
            for j in range(size):
                if (i-h)**2+(j-w)**2<r**2: img[i,j]=c
    elif cls == 1: img[h-r:h+r, w-r:w+r] = c
    elif cls == 2:
        for i in range(h-r,h+r):
            wid=int((i-(h-r))/(2*r)*2*r); img[i,w-wid:w+wid]=c
    else:
        for i in range(size):
            for j in range(size):
                if abs(i-h)+abs(j-w)<r: img[i,j]=c
    img += (np.random.randn(*img.shape)*noise).astype(np.float32)
    return np.clip(img,0,1)

class PatchEmbed(nn.Module):
    def __init__(self, patch_size=4, in_ch=3, embed_dim=48):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, embed_dim, patch_size, patch_size)
    def forward(self, x): return self.proj(x).flatten(2).transpose(1,2)

class MiniVLM(nn.Module):
    """Tiny VLM: ViT image encoder + GPT text decoder."""
    def __init__(self, vocab_size=16, embed_dim=48, n_heads=3, n_layers=2):
        super().__init__()
        self.n_patches = (32//4)**2  # 64
        self.patch_embed = PatchEmbed(embed_dim=embed_dim)
        self.img_pos = nn.Parameter(torch.zeros(1, self.n_patches, embed_dim))
        enc_layer = nn.TransformerEncoderLayer(embed_dim, n_heads,
                        dim_feedforward=embed_dim*4, batch_first=True, dropout=0.0)
        self.img_encoder = nn.TransformerEncoder(enc_layer, n_layers)
        self.tok_emb = nn.Embedding(vocab_size, embed_dim)
        self.txt_pos = nn.Embedding(32, embed_dim)
        dec_layer = nn.TransformerDecoderLayer(embed_dim, n_heads,
                        dim_feedforward=embed_dim*4, batch_first=True, dropout=0.0)
        self.decoder = nn.TransformerDecoder(dec_layer, n_layers)
        self.head = nn.Linear(embed_dim, vocab_size)
        nn.init.trunc_normal_(self.img_pos, std=0.02)
    def encode_image(self, img):
        x = self.patch_embed(img) + self.img_pos
        return self.img_encoder(x)  # (B, n_patches, embed_dim)
    def forward(self, img, tgt):
        memory = self.encode_image(img)
        T = tgt.size(1)
        pos = torch.arange(T, device=tgt.device).unsqueeze(0)
        x = self.tok_emb(tgt) + self.txt_pos(pos)
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=tgt.device)
        x = self.decoder(x, memory, tgt_mask=mask)
        return self.head(x)  # (B, T, vocab_size)

n_params = sum(p.numel() for p in MiniVLM().parameters())
print(f"MiniVLM defined — {n_params:,} parameters")